# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
%load_ext dotenv
%dotenv 


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [10]:
import os
from glob import glob

# Write your code below.
price_data = os.getenv('PRICE_DATA')

parquet_files = glob(os.path.join(price_data, '**', '*.parquet'), recursive=True)
print(f"Searching for parquet files in: {os.path.join(price_data, '**', '*.parquet')}")
print(f"Found {len(parquet_files)} parquet files.")


Searching for parquet files in: ../../05_src/data/prices/**/*.parquet
Found 3144 parquet files.


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.

# Load all parquet files as a Dask DataFrame
ddf = dd.read_parquet(parquet_files)

# Add lags and features using groupby and assign
dd_feat = ddf.groupby('ticker', group_keys=False).apply(
    lambda df: df.assign(
        Close_lag_1 = df['Close'].shift(1),
        Adj_Close_lag_1 = df['Adj Close'].shift(1),
        returns = df['Close'] / df['Close'].shift(1) - 1,
        hi_lo_range = df['High'] - df['Low']
    )
)

dd_feat = dd_feat.persist() 

# Show a sample
dd_feat.head()

/var/folders/r3/cnfgms0529qcjggnzd7h1jjc0000gn/T/ipykernel_35624/2352536316.py:7: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = ddf.groupby('ticker', group_keys=False).apply(


,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
178245,2001-06-29,1.959375,2.048438,1.950000,2.031250,0.159913,1238400.0,INFY.csv,INFY,2001,NaN,NaN,NaN,0.098438
178246,2001-07-02,1.996562,2.031250,1.903125,1.940625,0.152778,2864000.0,INFY.csv,INFY,2001,2.031250,0.159913,-0.044615,0.128125
178247,2001-07-03,1.890625,1.900000,1.834375,1.837500,0.144659,2649600.0,INFY.csv,INFY,2001,1.940625,0.152778,-0.053140,0.065625
178248,2001-07-05,1.834375,1.837500,1.802187,1.818750,0.143183,3120000.0,INFY.csv,INFY,2001,1.837500,0.144659,-0.010204,0.035313
178249,2001-07-06,1.823125,1.834062,1.805625,1.805938,0.142175,2054400.0,INFY.csv,INFY,2001,1.818750,0.143183,-0.007045,0.028437


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [12]:
# Write your code below.

# Dask DF to pandas DF
df_feat = dd_feat.compute()

# 10-day moving average of returns
df_feat['returns_ma_10'] = df_feat.groupby('ticker')['returns'].transform(lambda x: x.rolling(10).mean())

# Show a sample
df_feat.head()


,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
178245,2001-06-29,1.959375,2.048438,1.950000,2.031250,0.159913,1238400.0,INFY.csv,INFY,2001,NaN,NaN,NaN,0.098438,NaN
178246,2001-07-02,1.996562,2.031250,1.903125,1.940625,0.152778,2864000.0,INFY.csv,INFY,2001,2.031250,0.159913,-0.044615,0.128125,NaN
178247,2001-07-03,1.890625,1.900000,1.834375,1.837500,0.144659,2649600.0,INFY.csv,INFY,2001,1.940625,0.152778,-0.053140,0.065625,NaN
178248,2001-07-05,1.834375,1.837500,1.802187,1.818750,0.143183,3120000.0,INFY.csv,INFY,2001,1.837500,0.144659,-0.010204,0.035313,NaN
178249,2001-07-06,1.823125,1.834062,1.805625,1.805938,0.142175,2054400.0,INFY.csv,INFY,2001,1.818750,0.143183,-0.007045,0.028437,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

Answer: 
1. It was not strictly necessary. Dask DFs also support rolling window operations but with some limitations. 
2. It is better to use Dask for the moving average calculation to take advantage of parallel and out-of-core computation. However, for smaller datasets or when you need more flexibility and compatibility with pandas features, converting to pandas is simpler and sometimes more convenient

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [x] Created a branch with the correct naming convention.
- [x] Ensured that the repository is public.
- [x] Reviewed the PR description guidelines and adhered to them.
- [x] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.